# 06. 관세청 API — flow0 / flow2 수집

## 목적
관세청 `품목별 국가별 수출입실적(GW)` API를 사용해 삼각무역 탐지에 필요한 두 가지 흐름을 수집한다.

- **flow0**: 규제국 → 한국 (규제 품목의 규제국으로부터 한국 수입)
- **flow2**: 후보국 → 한국 (규제 품목의 제3국으로부터 한국 수입)

두 흐름 모두 "한국 입장에서의 수입" 데이터이므로, **동일한 API 호출 결과**에서 분리할 수 있다.  
→ HS 코드 + 기간만 고정하고 국가 필터 없이 호출 → 규제국 행 = flow0, 나머지 행 = flow2 후보

## 입력
- `data/interim/regulation_events_atomic(~2015).csv`

## 출력
- `data/interim/customs_flow0_flow2_raw.csv`

## 0. 라이브러리 & 설정

In [21]:
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import requests

from src.config import get_customs_api_key

In [22]:
# --- 경로 설정 ---
DATA_DIR = Path("../data/interim")
INPUT_PATH = DATA_DIR / "regulation_events_atomic(~2015).csv"
OUTPUT_PATH = DATA_DIR / "customs_flow0_flow2_raw.csv"

# API 호출 결과를 캐싱해 재실행 시 불필요한 API 호출을 방지한다
CACHE_DIR = DATA_DIR / "cache" / "customs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- API 설정 ---
API_URL = "https://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList"
SERVICE_KEY = get_customs_api_key()

# API 연속 호출 시 서버 부하를 줄이기 위한 대기 시간 (초)
SLEEP_BETWEEN_CALLS = 0.5

## 1. 국가 코드 매핑 (ISO3 → ISO2)

관세청 API는 **2자리 ISO 국가 코드(ISO 3166-1 alpha-2)** 를 사용한다.  
원자화 파일의 `origin_country_iso3` 컬럼은 **3자리 코드(ISO 3166-1 alpha-3)** 이므로 변환이 필요하다.

규제 이벤트에 등장하는 19개 국가만 수동으로 정의한다.

In [23]:
# 규제 이벤트에 등장하는 국가만 포함 (ISO3 → ISO2)
ISO3_TO_ISO2 = {
    "ARE": "AE",  # 아랍에미리트
    "AUS": "AU",  # 호주
    "CHN": "CN",  # 중국
    "EGY": "EG",  # 이집트
    "ESP": "ES",  # 스페인
    "FIN": "FI",  # 핀란드
    "FRA": "FR",  # 프랑스
    "IDN": "ID",  # 인도네시아
    "IND": "IN",  # 인도
    "ITA": "IT",  # 이탈리아
    "JPN": "JP",  # 일본
    "MYS": "MY",  # 말레이시아
    "SAU": "SA",  # 사우디아라비아
    "SGP": "SG",  # 싱가포르
    "THA": "TH",  # 태국
    "TWN": "TW",  # 대만
    "UKR": "UA",  # 우크라이나
    "USA": "US",  # 미국
    "VNM": "VN",  # 베트남
}

## 2. API 호출 함수 정의

In [24]:
def call_customs_api(hs_code: str, start_yymm: str, end_yymm: str, country_iso2: str = "") -> list[dict]:
    """
    관세청 수출입실적 API 단건 호출 (최대 12개월 범위).

    Parameters
    ----------
    hs_code : str
        HS 품목 코드 (예: "700529")
    start_yymm : str
        조회 시작 연월, YYYYMM 형식 (예: "201401")
    end_yymm : str
        조회 종료 연월, YYYYMM 형식 (예: "201412")
        start_yymm과의 차이가 12개월을 초과하면 안 됨
    country_iso2 : str, optional
        2자리 국가 코드. 빈 문자열이면 모든 국가 데이터를 반환.

    Returns
    -------
    list[dict]
        월별 수출입 실적 행 목록. 각 행은 아래 키를 포함:
        trade_date, country_iso2, country_name_kr, hs_code,
        imp_wgt(수입중량 kg), imp_dlr(수입금액 달러)
    """
    params = {
        "serviceKey": SERVICE_KEY,
        "strtYymm": start_yymm,
        "endYymm": end_yymm,
        "hsSgn": hs_code,
        # cntyCd가 빈 문자열이면 API가 모든 국가 데이터를 반환한다
        "cntyCd": country_iso2,
    }

    res = requests.get(API_URL, params=params, timeout=30)

    if res.status_code != 200:
        raise RuntimeError(f"HTTP {res.status_code}: {res.text[:200]}")

    root = ET.fromstring(res.text)

    result_code = root.findtext("./header/resultCode")
    result_msg = root.findtext("./header/resultMsg")

    if result_code != "00":
        raise RuntimeError(f"API 오류 [{result_code}]: {result_msg}")

    rows = []
    for item in root.findall("./body/items/item"):
        # '총계' 행은 월별 집계가 아니라 전체 합산 행이므로 제외
        if item.findtext("year") == "총계":
            continue

        def safe_int(tag: str) -> int | None:
            """XML 태그 값을 int로 변환. 값이 없거나 '-'이면 None 반환."""
            v = item.findtext(tag)
            if v in (None, "", "-"):
                return None
            try:
                return int(v)
            except ValueError:
                return None

        # API의 year 필드는 'YYYYMM' 형식 → 'YYYY-MM'으로 변환해 trade_date로 사용
        raw_date = item.findtext("year", "")
        trade_date = f"{raw_date[:4]}-{raw_date[4:]}" if len(raw_date) == 6 else raw_date

        rows.append({
            "trade_date": trade_date,
            "country_iso2": item.findtext("statCd"),
            "country_name_kr": item.findtext("statCdCntnKor1"),
            "hs_code": item.findtext("hsCd"),
            "imp_wgt": safe_int("impWgt"),   # 수입 중량 (kg)
            "imp_dlr": safe_int("impDlr"),   # 수입 금액 (달러)
        })

    return rows

In [25]:
def collect_customs_for_event(
    hs_code: str,
    window_start: str,
    window_end: str,
) -> pd.DataFrame:
    """
    규제 이벤트 1건에 대해 전체 수집 기간의 관세청 데이터를 수집한다.

    관세청 API는 한 번 호출에 최대 12개월만 조회 가능하다.
    수집 기간(window)이 최대 25개월(±12개월)이므로, 12개월 단위로 분할해 호출한다.

    Parameters
    ----------
    hs_code : str
        HS 품목 코드
    window_start : str
        수집 시작 연월, 'YYYY-MM' 형식
    window_end : str
        수집 종료 연월, 'YYYY-MM' 형식

    Returns
    -------
    pd.DataFrame
        수집 기간 전체의 월별 수출입 실적
    """
    # 'YYYY-MM' → 'YYYYMM'으로 변환 (API 요청 형식)
    start_dt = pd.to_datetime(window_start + "-01")
    end_dt = pd.to_datetime(window_end + "-01")

    # 12개월 단위로 기간 분할
    periods = []
    cursor = start_dt
    while cursor <= end_dt:
        chunk_end = cursor + pd.DateOffset(months=11)
        if chunk_end > end_dt:
            chunk_end = end_dt
        periods.append((
            cursor.strftime("%Y%m"),
            chunk_end.strftime("%Y%m"),
        ))
        cursor = chunk_end + pd.DateOffset(months=1)

    all_rows = []
    for s, e in periods:
        # cntyCd 없이 호출 → 모든 국가의 데이터를 한 번에 수집
        rows = call_customs_api(hs_code=hs_code, start_yymm=s, end_yymm=e)
        all_rows.extend(rows)
        time.sleep(SLEEP_BETWEEN_CALLS)

    return pd.DataFrame(all_rows)

## 3. API 동작 테스트

본격 수집 전, 단일 이벤트로 API 응답을 확인한다.

> **확인 포인트**
> - `cntyCd` 없이 호출했을 때 모든 국가 데이터가 반환되는지
> - 응답 컬럼 형식이 예상과 일치하는지

In [26]:
# 테스트용: AD-01-01 (플로트판유리, HS 700529, 중국, 2015-01-07 규제 시작)
# 수집 기간: 2014-01 ~ 2016-01
df_test = collect_customs_for_event(
    hs_code="700529",
    window_start="2014-01",
    window_end="2016-01",
)

print("수집 행 수:", len(df_test))
print("국가 수:", df_test["country_iso2"].nunique())
print("\n국가 목록:")
print(df_test[["country_iso2", "country_name_kr"]].drop_duplicates().to_string())

수집 행 수: 1670
국가 수: 41

국가 목록:
     country_iso2 country_name_kr
0              AE       아랍에미리트 연합
1              CN              중국
19             DE              독일
23             EG             이집트
25             ES             스페인
28             HK              홍콩
29             ID           인도네시아
36             JP              일본
39             LU           룩셈부르그
40             MY           말레이시아
41             SA         사우디아라비아
47             TH              태국
53             TW              대만
63             US              미국
65             VN             베트남
94             FR             프랑스
102            IN              인도
122            TR            튀르키예
167            GT            과테말라
180            PH             필리핀
203            UZ          우즈베키스탄
250            RO            루마니아
300            EC           에쿠아도르
301            GB              영국
309            IL            이스라엘
310            IT            이탈리아
312            RU          러시아 연방
339            AO 

In [27]:
# 중국(CN) 행 = flow0, 나머지 = flow2 후보인지 확인
df_test.head(10)

,trade_date,country_iso2,country_name_kr,hs_code,imp_wgt,imp_dlr
0,2014.01,AE,아랍에미리트 연합,7005294090,38686,11235
1,2014.01,CN,중국,7005291010,0,0
2,2014.01,CN,중국,7005291091,0,0
3,2014.01,CN,중국,7005291099,27925,153473
4,2014.01,CN,중국,7005292010,592320,216556
5,2014.01,CN,중국,7005292020,3,2835
6,2014.01,CN,중국,7005292091,1018556,522135
7,2014.01,CN,중국,7005292099,89878,29918
8,2014.01,CN,중국,7005293010,1045307,449918
9,2014.01,CN,중국,7005293020,372257,179047


## 4. 전체 이벤트 수집

원자화 파일의 모든 행에 대해 수집을 수행한다.

- 동일한 `(hs_code, window_start, window_end)` 조합은 중복 호출을 막기 위해 **한 번만** 호출한다.
- 수집 결과는 `data/interim/cache/customs/` 에 parquet 파일로 캐싱된다.
  - **재실행 시 캐시 파일이 존재하면 API를 호출하지 않는다.**
  - 기존 수집 결과가 있다면 아래 **캐시 시딩 셀**을 먼저 실행해 캐시를 채울 수 있다.
- 수집 결과는 `event_id`를 포함해 이후 flow 분리 단계에서 식별할 수 있도록 저장한다.

In [28]:
df_atomic = pd.read_csv(INPUT_PATH)
print("원자화 이벤트 수:", len(df_atomic))
df_atomic[["event_id", "origin_country_iso3", "hs_code", "window_start", "window_end"]].head()

원자화 이벤트 수: 155


,event_id,origin_country_iso3,hs_code,window_start,window_end
0,AD-01-01,CHN,700529,2014-01,2016-01
1,AD-02-01,CHN,690721,2014-02,2016-02
2,AD-02-01,CHN,690722,2014-02,2016-02
3,AD-02-01,CHN,690723,2014-02,2016-02
4,AD-03-01,CHN,721633,2014-07,2016-07


In [29]:
# 중복 호출 방지: (hs_code, window_start, window_end)가 같은 행은 동일한 API 결과를 반환하므로
# 고유한 조합만 추출한 뒤 수집하고, 나중에 event_id와 join한다.
call_keys = (
    df_atomic
    .groupby(["hs_code", "window_start", "window_end"])
    .agg(event_ids=("event_id", list),
         regulated_iso3s=("origin_country_iso3", list))
    .reset_index()
)

print(f"고유 API 호출 조합: {len(call_keys)}건 (원본 {len(df_atomic)}행에서 중복 제거)")
call_keys.head()

고유 API 호출 조합: 84건 (원본 155행에서 중복 제거)


,hs_code,window_start,window_end,event_ids,regulated_iso3s
0,4412,2016-05,2018-05,"[AD-11-01, AD-12-01]","[MYS, CHN]"
1,4412,2019-11,2021-11,"[AD-30-01, AD-31-01, AD-32-01]","[MYS, CHN, VNM]"
2,4412,2023-07,2025-07,"[AD-50-01, AD-52-01, AD-53-01]","[CHN, MYS, VNM]"
3,7219,2020-09,2022-09,"[AD-36-01, AD-36-02, AD-36-03]","[CHN, IDN, TWN]"
4,7219,2024-05,2026-05,"[AD-56-01, AD-56-02, AD-56-03]","[CHN, IDN, TWN]"


In [30]:
# [선택] 기존 수집 결과가 있으면 캐시를 미리 채운다 (API 재호출 방지)
# 이미 customs_flow0_flow2_raw.csv가 존재하는 경우 한 번만 실행하면 된다.
RAW_COLS = ["trade_date", "country_iso2", "country_name_kr", "hs_code", "imp_wgt", "imp_dlr"]

if OUTPUT_PATH.exists():
    _df_existing = pd.read_csv(OUTPUT_PATH)
    seeded = 0
    for (hs_q, ws, we), grp in _df_existing.groupby(["hs_code_query", "window_start", "window_end"]):
        cache_path = CACHE_DIR / f"hs{hs_q}_{ws}_{we}.parquet"
        if not cache_path.exists():
            grp[RAW_COLS].drop_duplicates().to_parquet(cache_path, index=False)
            seeded += 1
    print(f"캐시 시딩 완료: {seeded}건 신규 저장 (이미 존재하는 파일은 건너뜀)")
else:
    print("기존 수집 파일 없음 — 수집 셀 실행 시 API를 호출합니다.")

캐시 시딩 완료: 79건 신규 저장 (이미 존재하는 파일은 건너뜀)


In [31]:
results = []  # 수집된 모든 행을 누적할 리스트
failed = []   # 호출 실패한 조합을 기록 (재시도 또는 디버깅용)

total = len(call_keys)

for i, row in call_keys.iterrows():
    hs_code = str(row["hs_code"])
    window_start = row["window_start"]
    window_end = row["window_end"]

    # 캐시 파일 경로: hs{code}_{window_start}_{window_end}.parquet
    cache_path = CACHE_DIR / f"hs{hs_code}_{window_start}_{window_end}.parquet"

    print(f"[{i+1}/{total}] HS {hs_code} | {window_start} ~ {window_end}", end=" ... ")

    # 캐시가 존재하면 API 호출 없이 로드
    if cache_path.exists():
        df_chunk = pd.read_parquet(cache_path)
        print(f"{len(df_chunk)}행 (캐시)")
    else:
        try:
            df_chunk = collect_customs_for_event(
                hs_code=hs_code,
                window_start=window_start,
                window_end=window_end,
            )

            if df_chunk.empty:
                print("데이터 없음")
                # 빈 결과도 캐싱해 다음 실행 시 불필요한 API 호출을 방지
                df_chunk.to_parquet(cache_path, index=False)
                continue
            else:
                df_chunk.to_parquet(cache_path, index=False)
                print(f"{len(df_chunk)}행 수집")

        except Exception as e:
            print(f"실패 — {e}")
            failed.append({
                "hs_code": hs_code,
                "window_start": window_start,
                "window_end": window_end,
                "error": str(e),
            })
            continue

    if not df_chunk.empty:
        # 어느 이벤트의 수집 결과인지 추적하기 위해 메타 컬럼을 붙인다
        df_chunk = df_chunk.copy()
        df_chunk["hs_code_query"] = hs_code
        df_chunk["window_start"] = window_start
        df_chunk["window_end"] = window_end
        df_chunk["regulated_iso3s"] = str(row["regulated_iso3s"])
        results.append(df_chunk)

print(f"\n수집 완료: 성공 {total - len(failed)}건 / 실패 {len(failed)}건")
print(f"캐시 저장 위치: {CACHE_DIR}")

[1/84] HS 4412 | 2016-05 ~ 2018-05 ... 1858행 (캐시)
[2/84] HS 4412 | 2019-11 ~ 2021-11 ... 1594행 (캐시)
[3/84] HS 4412 | 2023-07 ~ 2025-07 ... 1710행 (캐시)
[4/84] HS 7219 | 2020-09 ~ 2022-09 ... 4888행 (캐시)
[5/84] HS 7219 | 2024-05 ~ 2026-05 ... 4360행 (캐시)
[6/84] HS 252321 | 2023-04 ~ 2025-04 ... 163행 (캐시)
[7/84] HS 281830 | 2022-04 ~ 2024-04 ... 602행 (캐시)
[8/84] HS 283110 | 2024-12 ~ 2026-12 ... 302행 (캐시)
[9/84] HS 290943 | 2015-12 ~ 2017-12 ... 457행 (캐시)
[10/84] HS 290943 | 2021-07 ~ 2023-07 ... 476행 (캐시)
[11/84] HS 290943 | 2021-09 ~ 2023-09 ... 482행 (캐시)
[12/84] HS 291531 | 2014-11 ~ 2016-11 ... 327행 (캐시)
[13/84] HS 291531 | 2018-07 ~ 2020-07 ... 409행 (캐시)
[14/84] HS 292211 | 2017-08 ~ 2019-08 ... 256행 (캐시)
[15/84] HS 292212 | 2017-08 ~ 2019-08 ... 250행 (캐시)
[16/84] HS 292213 | 2017-08 ~ 2019-08 ... 데이터 없음
[17/84] HS 370130 | 2016-09 ~ 2018-09 ... 716행 (캐시)
[18/84] HS 370130 | 2020-05 ~ 2022-05 ... 689행 (캐시)
[19/84] HS 370130 | 2021-10 ~ 2023-10 ... 680행 (캐시)
[20/84] HS 390760 | 2023-11 ~

In [34]:
# 실패 목록 확인
if failed:
    print("실패한 호출:")
    for f in failed:
        print(f)

## 5. flow0 / flow2 분리 및 병합

수집된 전체 데이터에서:
- 해당 이벤트의 **규제국**에 해당하는 행 → `flow = 0`
- 나머지 국가 행 → `flow = 2` (우회 후보국, 이후 flow1 수집 대상)

In [35]:
if not results:
    raise RuntimeError("수집된 데이터가 없습니다. 위 수집 셀을 먼저 실행하세요.")

df_raw = pd.concat(results, ignore_index=True)
print("전체 수집 행 수:", len(df_raw))
df_raw.head()

전체 수집 행 수: 68712


,trade_date,country_iso2,country_name_kr,hs_code,imp_wgt,imp_dlr,hs_code_query,window_start,window_end,regulated_iso3s
0,2016.05,BA,보스니아-헤르체고비나,441299,192,703,4412,2016-05,2018-05,"['MYS', 'CHN']"
1,2016.05,BE,벨기에,441299,31250,119764,4412,2016-05,2018-05,"['MYS', 'CHN']"
2,2016.05,BN,브루나이,441299,0,0,4412,2016-05,2018-05,"['MYS', 'CHN']"
3,2016.05,BR,브라질,441239,24930,18372,4412,2016-05,2018-05,"['MYS', 'CHN']"
4,2016.05,BR,브라질,441299,0,0,4412,2016-05,2018-05,"['MYS', 'CHN']"


In [36]:
# regulated_iso3s 컬럼은 문자열로 저장된 리스트("['CHN', 'JPN']") → 실제 집합으로 파싱
import ast

# ISO3 → ISO2 변환 함수
def iso3_list_to_iso2_set(iso3_str: str) -> set[str]:
    """문자열로 직렬화된 ISO3 코드 리스트를 ISO2 집합으로 변환."""
    iso3_list = ast.literal_eval(iso3_str)
    return {ISO3_TO_ISO2[c] for c in iso3_list if c in ISO3_TO_ISO2}


# 각 행의 country_iso2가 해당 이벤트의 규제국인지 판별 → flow 태깅
def assign_flow(row: pd.Series) -> int:
    """
    규제국 ISO2 집합에 현재 행의 country_iso2가 포함되면 flow=0(규제국→한국),
    포함되지 않으면 flow=2(후보국→한국).
    """
    regulated_iso2s = iso3_list_to_iso2_set(row["regulated_iso3s"])
    return 0 if row["country_iso2"] in regulated_iso2s else 2


df_raw["flow"] = df_raw.apply(assign_flow, axis=1)

print("flow 분포:")
print(df_raw["flow"].value_counts())

flow 분포:
flow
2    60520
0     8192
Name: count, dtype: int64


In [37]:
# 원자화 파일과 조인해 event_id, source_row_id, 규제 기간 등 메타 컬럼 복원
df_meta = df_atomic[[
    "event_id", "source_row_id", "hs_code",
    "origin_country_name_kr", "origin_country_iso3",
    "product_name_kr",
    "start_date", "end_date",
    "window_start", "window_end",
]].copy()

# CSV에서 읽힌 hs_code는 int64, df_raw의 hs_code_query는 str → 타입 통일
df_meta["hs_code"] = df_meta["hs_code"].astype(str)
df_meta = df_meta.rename(columns={"hs_code": "hs_code_query"})

df_final = df_raw.merge(
    df_meta,
    on=["hs_code_query", "window_start", "window_end"],
    how="left",
)

print("최종 행 수:", len(df_final))
df_final.head()

최종 행 수: 141907


,trade_date,country_iso2,country_name_kr,hs_code,imp_wgt,imp_dlr,hs_code_query,window_start,window_end,regulated_iso3s,flow,event_id,source_row_id,origin_country_name_kr,origin_country_iso3,product_name_kr,start_date,end_date
0,2016.05,BA,보스니아-헤르체고비나,441299,192,703,4412,2016-05,2018-05,"['MYS', 'CHN']",2,AD-11-01,11,말레이시아,MYS,합판(2차재심),2017-05-08,2020-05-07
1,2016.05,BA,보스니아-헤르체고비나,441299,192,703,4412,2016-05,2018-05,"['MYS', 'CHN']",2,AD-12-01,12,중국,CHN,합판(1차재심),2017-05-08,2020-05-07
2,2016.05,BE,벨기에,441299,31250,119764,4412,2016-05,2018-05,"['MYS', 'CHN']",2,AD-11-01,11,말레이시아,MYS,합판(2차재심),2017-05-08,2020-05-07
3,2016.05,BE,벨기에,441299,31250,119764,4412,2016-05,2018-05,"['MYS', 'CHN']",2,AD-12-01,12,중국,CHN,합판(1차재심),2017-05-08,2020-05-07
4,2016.05,BN,브루나이,441299,0,0,4412,2016-05,2018-05,"['MYS', 'CHN']",2,AD-11-01,11,말레이시아,MYS,합판(2차재심),2017-05-08,2020-05-07


## 6. 결과 확인 및 저장

In [38]:
print("=== 기본 검증 ===")
print("Shape:", df_final.shape)
print("\nNull counts:")
print(df_final[["event_id", "trade_date", "country_iso2", "imp_wgt", "imp_dlr", "flow"]].isnull().sum())
print("\nflow 분포:")
print(df_final["flow"].value_counts())

=== 기본 검증 ===
Shape: (141907, 18)

Null counts:
event_id         0
trade_date       0
country_iso2    29
imp_wgt          0
imp_dlr          0
flow             0
dtype: int64

flow 분포:
flow
2    123928
0     17979
Name: count, dtype: int64


In [39]:
# 샘플 확인: AD-01-01 (플로트판유리, 규제국 중국)
# flow=0이면 중국(CN), flow=2이면 그 외 국가여야 한다
sample = df_final[df_final["event_id"] == "AD-01-01"].copy()
print("flow=0 국가:", sample[sample["flow"] == 0]["country_iso2"].unique())
print("flow=2 국가:", sorted(sample[sample["flow"] == 2]["country_iso2"].unique()))

flow=0 국가: <ArrowStringArray>
['CN']
Length: 1, dtype: str
flow=2 국가: ['AE', 'AO', 'BE', 'BG', 'BR', 'CA', 'DE', 'EC', 'EG', 'ES', 'FR', 'GB', 'GT', 'HK', 'ID', 'IL', 'IN', 'IQ', 'IT', 'JP', 'LU', 'MN', 'MX', 'MY', 'NL', 'OM', 'PH', 'QA', 'RO', 'RU', 'RW', 'SA', 'TH', 'TM', 'TR', 'TW', 'US', 'UZ', 'VE', 'VN']


In [40]:
# 불필요한 내부 컬럼 제거 후 저장
df_save = df_final.drop(columns=["regulated_iso3s"], errors="ignore")

df_save.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}")
print(f"총 {len(df_save)}행")

저장 완료: ../data/interim/customs_flow0_flow2_raw.csv
총 141907행
